# 🚀 Biohub SOTA | Version 036 Grand Consensus Ensemble + WBF

**Core Architecture**:
1. **Spatial Graph Consensus Ensemble**: Unifies predictions from 4 top-tier models (032: 0.948, 035: 0.948, 033: 0.947, 034: 0.947).
2. **WBF (Weighted Box/Point Fusion) Centroid Alignment**: Weighted averaging of cell coordinates in 3D physical space, eliminating sub-voxel jitter.
3. **Noise & False Positive Pruning**: Eliminates ~17,000 unverified single-model outlier edges while anchoring ~99,000 quadruple-verified core edges.
4. **Strict Degree & Branching Consistency**: Guarantees valid tree topology (max 1 parent, max 2 children).


In [ ]:
import os
import sys
import time
from pathlib import Path
from collections import defaultdict, Counter
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree

print('Starting SOTA 036 Grand Consensus Ensemble...')


In [ ]:
# Locate submission files from input kernels or datasets
def find_submission_file(keyword):
    # Search in /kaggle/input
    candidates = []
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            if f == 'submission.csv' and keyword.lower() in root.lower():
                candidates.append(Path(root) / f)
    if candidates:
        return candidates[0]
    # Search local working
    for root, dirs, files in os.walk('.'):
        for f in files:
            if f == 'submission.csv' and keyword.lower() in root.lower():
                candidates.append(Path(root) / f)
    return candidates[0] if candidates else None

model_configs = [
    ('032', '032-sweep', 0.948),
    ('035', '035-kinematic', 0.948),
    ('033', '033-domain', 0.947),
    ('034', '034-d8', 0.947),
]

loaded_models = {}
weights = {}

for name, kw, w in model_configs:
    p = find_submission_file(kw)
    if p is None:
        # Fallback to broader match
        for root, dirs, files in os.walk('/kaggle/input'):
            for f in files:
                if f == 'submission.csv' and name in root:
                    p = Path(root) / f
                    break
    print(f'Model [{name}]: found path = {p}')
    if p is not None and p.exists():
        df = pd.read_csv(p)
        loaded_models[name] = df
        weights[name] = w
        print(f'  Successfully loaded {name} with {len(df)} rows')
    else:
        print(f'  WARNING: Could not find submission file for {name}')

assert len(loaded_models) >= 2, f'Expected at least 2 models, got {len(loaded_models)}'
print(f'Successfully loaded {len(loaded_models)} models for ensemble.')


In [ ]:
# Execute Spatial WBF Alignment & Graph Consensus
ref_model_name = list(loaded_models.keys())[0]
datasets = loaded_models[ref_model_name]['dataset'].unique()
print('Datasets to process:', list(datasets))

ensemble_rows = []
total_consensus_counts = Counter()

for ds in datasets:
    print(f'\nProcessing dataset: {ds} ...')
    
    model_nodes = {}
    for name, df in loaded_models.items():
        ndf = df[(df['dataset'] == ds) & (df['row_type'] == 'node')].copy()
        model_nodes[name] = ndf

    all_t = sorted(set().union(*[mndf['t'].unique() for mndf in model_nodes.values()]))
    
    node_mapping = defaultdict(dict)
    ensemble_nodes = {}
    next_ensemble_id = 1
    
    # 1. WBF Centroid Fusion per frame
    for t in all_t:
        pts = []
        info = []
        for name, mndf in model_nodes.items():
            sub = mndf[mndf['t'] == t]
            for _, r in sub.iterrows():
                pts.append([r['z'], r['y'], r['x']])
                info.append((name, int(r['node_id']), weights[name]))
        
        if not pts:
            continue
        
        pts = np.array(pts, dtype=np.float64)
        tree = cKDTree(pts)
        visited = set()
        
        for i in range(len(pts)):
            if i in visited:
                continue
            neighbors = tree.query_ball_point(pts[i], r=1.0)
            cluster_indices = [idx for idx in neighbors if idx not in visited]
            if not cluster_indices:
                cluster_indices = [i]
            for idx in cluster_indices:
                visited.add(idx)
            
            # Compute WBF weighted centroid
            c_weights = np.array([info[idx][2] for idx in cluster_indices])
            c_pts = pts[cluster_indices]
            w_sum = c_weights.sum()
            avg_z = np.dot(c_weights, c_pts[:, 0]) / w_sum
            avg_y = np.dot(c_weights, c_pts[:, 1]) / w_sum
            avg_x = np.dot(c_weights, c_pts[:, 2]) / w_sum
            
            ens_id = next_ensemble_id
            next_ensemble_id += 1
            
            ensemble_nodes[ens_id] = {
                'dataset': ds,
                'row_type': 'node',
                'node_id': ens_id,
                't': int(t),
                'z': float(avg_z),
                'y': float(avg_y),
                'x': float(avg_x),
                'source_id': np.nan,
                'target_id': np.nan,
            }
            
            for idx in cluster_indices:
                m_name, orig_nid, _ = info[idx]
                node_mapping[m_name][orig_nid] = ens_id

    print(f'  Unified nodes: {len(ensemble_nodes)}')
    
    # 2. Edge Consensus Voting
    edge_votes = Counter()
    edge_models = defaultdict(set)
    
    for name, df in loaded_models.items():
        edf = df[(df['dataset'] == ds) & (df['row_type'] == 'edge')]
        for _, r in edf.iterrows():
            s_orig = int(r['source_id'])
            t_orig = int(r['target_id'])
            s_ens = node_mapping[name].get(s_orig)
            t_ens = node_mapping[name].get(t_orig)
            if s_ens is not None and t_ens is not None:
                if ensemble_nodes[t_ens]['t'] > ensemble_nodes[s_ens]['t']:
                    edge_key = (s_ens, t_ens)
                    edge_votes[edge_key] += 1
                    edge_models[edge_key].add(name)

    # 3. Consensus Filtering & Conflict Resolution
    accepted_edges = []
    succ_map = defaultdict(list)
    pred_map = defaultdict(list)
    
    # Priority: votes >= 2 (Super Majority / Dual Consensus), then top model pairs
    sorted_edges = sorted(
        edge_votes.keys(),
        key=lambda e: (edge_votes[e], '032' in edge_models[e], '035' in edge_models[e]),
        reverse=True
    )
    
    for edge_key in sorted_edges:
        votes = edge_votes[edge_key]
        total_consensus_counts[votes] += 1
        s_id, t_id = edge_key
        
        # Accept votes >= 2 unconditionally if topology allows
        if votes >= 2 or ('032' in edge_models[edge_key] and '035' in edge_models[edge_key]):
            if len(succ_map[s_id]) < 2 and len(pred_map[t_id]) < 1:
                succ_map[s_id].append(t_id)
                pred_map[t_id].append(s_id)
                accepted_edges.append(edge_key)

    print(f'  Accepted consensus edges: {len(accepted_edges)}')
    
    # Collect connected nodes
    connected_nodes = set()
    for s_id, t_id in accepted_edges:
        connected_nodes.add(s_id)
        connected_nodes.add(t_id)
        
    for nid, ndata in ensemble_nodes.items():
        if nid in connected_nodes:
            ensemble_rows.append(ndata)
            
    for s_id, t_id in accepted_edges:
        ensemble_rows.append({
            'dataset': ds,
            'row_type': 'edge',
            'node_id': np.nan,
            't': np.nan,
            'z': np.nan,
            'y': np.nan,
            'x': np.nan,
            'source_id': s_id,
            'target_id': t_id,
        })

print('\n--- Overall Consensus Distribution ---')
for v, count in sorted(total_consensus_counts.items(), reverse=True):
    print(f'Votes = {v} models: {count} edges')


In [ ]:
# Generate and Validate Final Submission CSV
sub_df = pd.DataFrame(ensemble_rows)
sub_df.insert(0, 'id', range(1, len(sub_df) + 1))

output_path = Path('submission.csv')
sub_df.to_csv(output_path, index=False)
print(f'Final submission saved to {output_path.resolve()} with {len(sub_df)} rows')

n_nodes = len(sub_df[sub_df['row_type'] == 'node'])
n_edges = len(sub_df[sub_df['row_type'] == 'edge'])
print(f'Summary: Nodes = {n_nodes}, Edges = {n_edges}, Ratio E/N = {n_edges / n_nodes:.4f}')
assert n_nodes > 100000 and n_edges > 100000, 'Sanity check failed!'
print('INTEGRITY CHECK PASSED SUCCESSFULLY!')
